# 20 · Энкодер: OOF-обучение по фолдам

Обёртка над пакетом `rag_reliability.methods.encoder` (задача C2). Сплит читается из
`folds.json`; `split_samples` не вызывается — иначе числа энкодера окажутся несравнимы
со всем остальным.

Одно обучение на 8192 токенах — ~2 ч, полный 5-fold OOF — ~10 ч. **Прогон длиннее двух
часов через ноутбук запускать нельзя**: VM останавливается при простое. Полный OOF идёт
через `jobs/encoder_oof.yaml`; здесь — смоук и одиночные конфигурации на 512–2048 токенах.

Чекпоинты — на File Storage, `save_strategy` внутри CLI. Контроль схлопывания
(`const_share`) печатается после каждой эпохи: прошлый прогон схлопнулся в константу,
и вывод «длинный контекст вредит» был сделан именно на нём.

## 0. Конфигурация

In [ ]:
# ======================= КОНФИГУРАЦИЯ — правится только здесь =======================
BASE     = "/home/jupyter/filestore/neurodrive"   # File Storage: переживает рестарт VM
REPO     = f"{BASE}/rag-reliability"
REPO_URL = "https://github.com/MurkaSelebry/rag-reliability.git"
BRANCH   = "integration"                          # НЕ qwen7b-notebook: та ветка устарела
CACHE    = f"{BASE}/cache/m3_judge"               # кэш судьи -> прогон резюмируется
LOGS     = f"{BASE}/logs"
DATA     = "data/alfa.jsonl"                      # канонический корпус, 2233 кейса
FOLDS    = "data/splits/folds_alfa.json"          # сплит только отсюда, split_samples не вызываем
MODEL    = "Qwen/Qwen2.5-7B-Instruct"
API_BASE = "http://localhost:8000/v1"
# ===================================================================================

import os, subprocess

branch = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "--abbrev-ref", "HEAD"]
).decode().strip()
assert branch == BRANCH, (
    f"репозиторий на ветке {branch}, ожидалась {BRANCH}. Прогон с другой ветки несравним "
    "с остальными: перезапусти notebooks/00_setup.ipynb"
)
print("branch:", branch)

os.environ["HF_HOME"] = f"{BASE}/hf"
CHECKPOINTS = f"{BASE}/encoder_checkpoints"
os.makedirs(CHECKPOINTS, exist_ok=True)
assert os.path.isfile(f"{REPO}/{FOLDS}"), (
    f"нет {REPO}/{FOLDS}: запусти сначала notebooks/00_setup.ipynb"
)
print("checkpoints:", CHECKPOINTS)


## 1. Железо

In [ ]:
import torch, psutil

n = torch.cuda.device_count()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if n else 0.0
print(f"GPU {torch.cuda.get_device_name(0) if n else '—'} | VRAM {vram:.0f}GB | "
      f"RAM {psutil.virtual_memory().total / 1e9:.0f}GB")
assert n >= 1, "выбрана CPU-конфигурация: обучение энкодера требует GPU"
assert vram >= 30, (
    f"VRAM {vram:.0f} GB. Энкодер до 4096 токенов идёт на g1.1 (V100 32 GB), "
    "8192 токенов и flash-attention 2 требуют g2.1 (A100 80 GB) — "
    "см. docs/specs/90_DATASPHERE_runbook.md §1"
)


## 2. Смоук — ~3 мин

Полное обучение без смоука не запускается: правило репозитория, и оно уже спасало
GPU-часы. `--limit 40` прогоняет весь путь (фолды → обучение → `scores.jsonl` →
валидация ключей) на 40 кейсах.

In [ ]:
!cd {REPO} && python scripts/train_encoder_baseline.py --variant smoke \
    --data {DATA} --folds {FOLDS} --limit 40 \
    --max-length 512 --batch-size 2 --epochs 1 \
    --output-dir {CHECKPOINTS}/smoke \
    --predictions-output {BASE}/smoke/encoder/scores.jsonl


## 3. Одна конфигурация, 2048 токенов — ~1 ч

Пять фолдов OOF внутри CLI. `const_share` по каждой эпохе идёт в лог (INFO) и в
`encoder_diagnostics.json` рядом с артефактом.

In [ ]:
# ~1 ч на A100. Обрыв = потеря прогона: чекпоинты пишутся, но CLI резюме не поддерживает,
# поэтому всё, что дольше двух часов, идёт через jobs/encoder_oof.yaml.
!cd {REPO} && python scripts/train_encoder_baseline.py --variant len2048_lr2e-5 \
    --data {DATA} --folds {FOLDS} \
    --max-length 2048 --batch-size 1 --grad-accum 8 \
    --epochs 3 --learning-rate 2e-5 --pos-weight-mode balanced \
    --output-dir {CHECKPOINTS}/len2048_lr2e-5 \
    --predictions-output predictions/alfa/encoder/len2048_lr2e-5/scores.jsonl


## 4. Контроль схлопывания — глазами

`const_share` — доля самого частого решения. Всё, что выше 0.98, означает модель,
предсказывающую константу: метрика при базовой ставке 72% всё ещё выглядит прилично,
а сигнала нет.

In [ ]:
import json

diagnostics = json.load(open(
    f"{REPO}/predictions/alfa/encoder/len2048_lr2e-5/encoder_diagnostics.json",
    encoding="utf-8",
))
print(f"{'repeat':>6} {'fold':>4} {'epoch':>5} {'const_share':>12} {'entropy':>8}  degenerate")
for row in diagnostics["epochs"]:
    print(f"{row['repeat']:>6} {row['fold']:>4} {row['epoch']:>5} "
          f"{row['const_share']:>12.4f} {row['output_entropy']:>8.4f}  {row['is_degenerate']}")
print()
print("collapsed:", diagnostics["collapsed"], "| причина:", diagnostics["collapse_reason"],
      "| фолды:", diagnostics["collapsed_folds"])


## 5. Оценка и коммит

In [ ]:
!cd {REPO} && python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores predictions/alfa/encoder/len2048_lr2e-5/scores.jsonl \
    --score-expr "enc.logit" --faith-expr "enc.logit" --rel-expr "enc.logit" \
    --output predictions/alfa/encoder/len2048_lr2e-5/report.json


In [ ]:
!cd {REPO} && git add predictions/alfa/encoder/len2048_lr2e-5 && \
    git commit -m "results(encoder): OOF 2048 токенов на A100" && \
    git log --oneline -1


## 6. Полный OOF на 8192 токенах — только через Jobs

~10 ч. Через ноутбук не запускать: VM остановится при простое и прогон умрёт на
середине. Конфиг задания — `jobs/encoder_oof.yaml`:

```bash
pip install datasphere
datasphere project job execute -p <project-id> -c jobs/encoder_oof.yaml
```

Задание исполняется на отдельной VM: сессию JupyterLab можно закрыть.